In [1]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ============================================================
# Imports
# ============================================================

import os
import sys
import random
import shutil
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [3]:
# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random Seed : {SEED}")

Random Seed : 42


In [4]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/AI_Image_Detection")

RAW_DATASETS = PROJECT_DIR / "raw_datasets"

REAL_DATASET = RAW_DATASETS / "real_images_final"
AI_DATASET = RAW_DATASETS / "ai_images_final"

PROCESSED_DATASET = PROJECT_DIR / "processed_dataset_final"

UTILS_DIR = PROJECT_DIR / "utils"

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Project Directory :", PROJECT_DIR)
print("Raw Datasets      :", RAW_DATASETS)
print("Real Dataset      :", REAL_DATASET)
print("AI Dataset        :", AI_DATASET)
print("Processed Dataset :", PROCESSED_DATASET)
print("Utilities         :", UTILS_DIR)

PROJECT PATHS
Project Directory : /content/drive/MyDrive/AI_Image_Detection
Raw Datasets      : /content/drive/MyDrive/AI_Image_Detection/raw_datasets
Real Dataset      : /content/drive/MyDrive/AI_Image_Detection/raw_datasets/real_images_final
AI Dataset        : /content/drive/MyDrive/AI_Image_Detection/raw_datasets/ai_images_final
Processed Dataset : /content/drive/MyDrive/AI_Image_Detection/processed_dataset_final
Utilities         : /content/drive/MyDrive/AI_Image_Detection/utils


In [5]:
# ============================================================
# Import Utility Functions
# ============================================================

sys.path.append(str(PROJECT_DIR))

from utils.dataset_utils import *

print("Utility functions imported successfully.")

Utility functions imported successfully.


In [6]:
# ============================================================
# Create Project Directories
# ============================================================

directories = [
    PROCESSED_DATASET,
    PROJECT_DIR / "checkpoints",
    PROJECT_DIR / "logs",
    PROJECT_DIR / "reports",
    PROJECT_DIR / "results"
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("PROJECT DIRECTORIES CREATED")
print("=" * 70)

for directory in directories:
    print(f"{directory.name:<20}: {directory.exists()}")

PROJECT DIRECTORIES CREATED
processed_dataset_final: True
checkpoints         : True
logs                : True
reports             : True
results             : True


In [7]:
# ============================================================
# Verify Project Structure
# ============================================================

print("=" * 70)
print("PROJECT STRUCTURE VERIFICATION")
print("=" * 70)

paths = {
    "Project Directory": PROJECT_DIR,
    "Raw Datasets": RAW_DATASETS,
    "Real Dataset": REAL_DATASET,
    "AI Dataset": AI_DATASET,
    "Processed Dataset": PROCESSED_DATASET,
    "Utilities": UTILS_DIR,
}

for name, path in paths.items():
    print(f"{name:<20}: {path.exists()}")

PROJECT STRUCTURE VERIFICATION
Project Directory   : True
Raw Datasets        : True
Real Dataset        : True
AI Dataset          : True
Processed Dataset   : True
Utilities           : True


In [ ]:
# # ============================================================
# # Inspect Dataset Structure
# # ============================================================

# def show_structure(root, max_depth=3):
#     """
#     Displays folder structure up to a given depth.
#     """

#     root = Path(root)

#     print("=" * 80)
#     print(root.name.upper())
#     print("=" * 80)

#     for path in sorted(root.rglob("*")):

#         if not path.is_dir():
#             continue

#         depth = len(path.relative_to(root).parts)

#         if depth > max_depth:
#             continue

#         indent = "    " * depth
#         print(f"{indent}📁 {path.name}")

#     print()


# # ---------------- REAL ----------------

# show_structure(REAL_DATASET)

# # ---------------- AI ----------------

# show_structure(AI_DATASET)

REAL_IMAGES_FINAL
    📁 ffhq
    📁 lfw
        📁 lfw-deepfunneled
            📁 lfw-deepfunneled
    📁 utkface
        📁 UTKFace

AI_IMAGES_FINAL
    📁 anime
    📁 deepfake
        📁 Final Dataset
            📁 Fake
    📁 finetune
    📁 nano_banana
        📁 Nano Banana 2.0 Dataset
            📁 dataset
    📁 stable_diffusion
        📁 stable-diffusion-face-dataset
            📁 .git
            📁 .idea
            📁 1024
            📁 512
            📁 768
    📁 stylegan3



In [8]:
# ============================================================
# Dataset Sampling Configuration
# ============================================================

SAMPLING_CONFIG = {

    "real": {

        "ffhq": {
            "paths": [
                REAL_DATASET / "ffhq"
            ],
            "count": 10000,
            "oversample": False,
        },

        "utkface": {
            "paths": [
                REAL_DATASET / "utkface" / "UTKFace"
            ],
            "count": 2000,
            "oversample": False,
        },

        "lfw": {
            "paths": [
                REAL_DATASET
                / "lfw"
                / "lfw-deepfunneled"
                / "lfw-deepfunneled"
            ],
            "count": 4000,
            "oversample": False,
        },
    },

    "ai": {

        "stable_diffusion": {
            "paths": [
                RAW_DATASETS / "ai_images" / "stable_diffusion"
                / "stable-diffusion-face-dataset" / "1024",

                RAW_DATASETS / "ai_images" / "stable_diffusion"
                / "stable-diffusion-face-dataset" / "768",
            ],
            "count": 6000,
            "oversample": False,
        },

        "nano_banana": {
            "paths": [
                RAW_DATASETS / "ai_images"
                / "nano_banana"
                / "Nano Banana 2.0 Dataset"
                / "dataset",
            ],
            "count": 2000,
            "oversample": True,
        },

        "deepfake": {
            "paths": [
                RAW_DATASETS / "ai_images"
                / "deepfake_faces"
                / "Final Dataset"
                / "Fake",
            ],
            "count": 3000,
            "oversample": False,
        },

        "finetune": {
            "paths": [

                RAW_DATASETS / "ai_images"
                / "finetune"
                / "new dataset"
                / "train"
                / "fake",

                RAW_DATASETS / "ai_images"
                / "finetune"
                / "new dataset"
                / "valid"
                / "fake",
            ],
            "count": 1000,
            "oversample": True,
        },

        "stylegan3": {
            "paths": [
                RAW_DATASETS / "ai_images"
                / "imagefake_validation"
                / "stylegan3-60K",
            ],
            "count": 3000,
            "oversample": True,
        },

        "anime": {
            "paths": [
                RAW_DATASETS / "ai_images"
                / "anime"
                / "train"
                / "images",
            ],
            "count": 1000,
            "oversample": False,
        },
    },
}

In [9]:
# ============================================================
# Verify Sampling Configuration (Skip LFW Count)
# ============================================================

def verify_sampling_config(skip_lfw=True):

    print("=" * 80)
    print("REAL DATASETS")
    print("=" * 80)

    total_real = 0

    for dataset_name, config in SAMPLING_CONFIG["real"].items():

        print(f"\n{dataset_name.upper()}")
        print("-" * 40)

        if skip_lfw and dataset_name == "lfw":

            available = 13233      # Already verified earlier

            print("Available Images : 13,233 (cached)")
            print(f"Requested Images : {config['count']:,}")
            print(f"Oversample       : {config['oversample']}")
            print("Status           : OK (Skipped Scan)")

        else:

            images = get_all_images(config["paths"])

            available = len(images)
            requested = config["count"]

            print(f"Available Images : {available:,}")
            print(f"Requested Images : {requested:,}")
            print(f"Oversample       : {config['oversample']}")

            if available < requested and not config["oversample"]:
                print("Status           : ERROR")
            else:
                print("Status           : OK")

        total_real += config["count"]

    print("\n" + "=" * 80)
    print("AI DATASETS")
    print("=" * 80)

    total_ai = 0

    for dataset_name, config in SAMPLING_CONFIG["ai"].items():

        images = get_all_images(config["paths"])

        available = len(images)
        requested = config["count"]

        print(f"\n{dataset_name.upper()}")
        print("-" * 40)
        print(f"Available Images : {available:,}")
        print(f"Requested Images : {requested:,}")
        print(f"Oversample       : {config['oversample']}")

        if available < requested:
            if config["oversample"]:
                print("Status           : Oversampling Required")
            else:
                print("Status           : ERROR")
        else:
            print("Status           : OK")

        total_ai += requested

    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)
    print(f"Total Real Images : {total_real:,}")
    print(f"Total AI Images   : {total_ai:,}")
    print(f"Grand Total       : {total_real + total_ai:,}")

In [17]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

False


In [18]:
!pip install -q insightface
!pip install -q onnxruntime

In [19]:
import sys

sys.path.append(str(PROJECT_DIR))

from preprocessing_pipeline import *

In [20]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False
No GPU


In [21]:
import preprocessing_pipeline

print(preprocessing_pipeline.__file__)

/content/drive/MyDrive/AI_Image_Detection/preprocessing_pipeline.py


In [ ]:
from pathlib import Path

processed_root = PROCESSED_DATASET

print("=" * 70)
print("PROCESSED DATASET STATUS")
print("=" * 70)

splits = ["train", "val", "test"]
labels = ["real", "ai"]

grand_total = 0

for split in splits:

    print(f"\n{split.upper()}")

    split_total = 0

    for label in labels:

        folder = processed_root / split / label

        if folder.exists():
            count = len(list(folder.glob("*.png")))
        else:
            count = 0

        split_total += count

        print(f"{label:<5}: {count:,}")

    grand_total += split_total

    print(f"Total : {split_total:,}")

print("\n" + "=" * 70)
print(f"TOTAL PROCESSED IMAGES : {grand_total:,}")
print("=" * 70)

PROCESSED DATASET STATUS

TRAIN
real : 12,803
ai   : 4,182
Total : 16,985

VAL
real : 0
ai   : 0
Total : 0

TEST
real : 0
ai   : 0
Total : 0

TOTAL PROCESSED IMAGES : 16,985


In [22]:
# ============================================================
# Run Preprocessing Pipeline (Frozen & Resumable)
# ============================================================

import shutil
from pathlib import Path

print("=" * 80)
print("STARTING PREPROCESSING PIPELINE")
print("=" * 80)

# ============================================================
# Frozen Metadata Files
# ============================================================

SELECTED_RECORDS_CSV = PROCESSED_DATASET / "selected_records.csv"

TRAIN_RECORDS_CSV = PROCESSED_DATASET / "train_records.csv"
VAL_RECORDS_CSV   = PROCESSED_DATASET / "val_records.csv"
TEST_RECORDS_CSV  = PROCESSED_DATASET / "test_records.csv"

# ============================================================
# Stage 0 : Collect / Load Frozen Sample
# ============================================================

if SELECTED_RECORDS_CSV.exists():

    print("\nLoading frozen sampled records...")

    all_records = load_records_from_csv(
        SELECTED_RECORDS_CSV
    )

    print(f"Loaded {len(all_records):,} sampled images.")

else:

    print("\nNo frozen selection found.")
    print("Collecting images from datasets...")

    all_records = []

    # ---------------- REAL ----------------

    for dataset_name, config in SAMPLING_CONFIG["real"].items():

        print(f"\nCollecting {dataset_name.upper()}...")

        all_records.extend(

            collect_dataset_images(
                config=config,
                source_name=dataset_name,
                label="real",
                group_by_folder=(dataset_name == "lfw"),
            )

        )

    # ---------------- AI ----------------

    for dataset_name, config in SAMPLING_CONFIG["ai"].items():

        print(f"\nCollecting {dataset_name.upper()}...")

        all_records.extend(

            collect_dataset_images(
                config=config,
                source_name=dataset_name,
                label="ai",
                group_by_folder=False,
            )

        )

    save_records_to_csv(
        all_records,
        SELECTED_RECORDS_CSV,
    )

    print(f"\nFrozen selection saved:")
    print(SELECTED_RECORDS_CSV)

print("\n" + "=" * 80)
print(f"TOTAL SELECTED IMAGES : {len(all_records):,}")
print("=" * 80)


# ============================================================
# Stage 1 : Frozen Train / Val / Test Split
# ============================================================

if (
    TRAIN_RECORDS_CSV.exists()
    and VAL_RECORDS_CSV.exists()
    and TEST_RECORDS_CSV.exists()
):

    print("\nLoading frozen dataset splits...")

    train_records = load_records_from_csv(TRAIN_RECORDS_CSV)
    val_records   = load_records_from_csv(VAL_RECORDS_CSV)
    test_records  = load_records_from_csv(TEST_RECORDS_CSV)

else:

    print("\nGenerating Train / Val / Test split...")

    train_records, val_records, test_records = stratified_group_split(
        all_records
    )

    save_records_to_csv(
        train_records,
        TRAIN_RECORDS_CSV,
    )

    save_records_to_csv(
        val_records,
        VAL_RECORDS_CSV,
    )

    save_records_to_csv(
        test_records,
        TEST_RECORDS_CSV,
    )

    print("\nFrozen splits saved.")

print("\n" + "=" * 80)
print("DATASET SPLIT")
print("=" * 80)

print(f"Train Images      : {len(train_records):,}")
print(f"Validation Images : {len(val_records):,}")
print(f"Test Images       : {len(test_records):,}")

# ============================================================
# Process Dataset
# ============================================================

print("\n" + "=" * 80)
print("PROCESSING TRAIN SET")
print("=" * 80)

train_manifest = process_and_save_dataset(
    train_records,
    PROCESSED_DATASET,
    "train",
)

print("\n" + "=" * 80)
print("PROCESSING VALIDATION SET")
print("=" * 80)

val_manifest = process_and_save_dataset(
    val_records,
    PROCESSED_DATASET,
    "val",
)

print("\n" + "=" * 80)
print("PROCESSING TEST SET")
print("=" * 80)

test_manifest = process_and_save_dataset(
    test_records,
    PROCESSED_DATASET,
    "test",
)

print("\n" + "=" * 80)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"Train Manifest : {train_manifest}")
print(f"Val Manifest   : {val_manifest}")
print(f"Test Manifest  : {test_manifest}")

STARTING PREPROCESSING PIPELINE

No frozen selection found.

[ffhq] collected 52,001 images
Sampled 10,000 / 52,001

[utkface] collected 23,708 images
Sampled 2,000 / 23,708

[lfw] collected 13,233 images
Sampled 4,000 / 13,233

[stable_diffusion] collected 6,000 images
Sampled 6,000 / 6,000

[nano_banana] collected 1,067 images
Oversampling 1,067 → 2,000

[deepfake] collected 7,000 images
Sampled 3,000 / 7,000

[finetune] collected 121 images
Oversampling 121 → 1,000

[stylegan3] collected 2,425 images
Oversampling 2,425 → 3,000

[anime] collected 5,293 images
Sampled 1,000 / 5,293

Frozen selection saved:
/content/drive/MyDrive/AI_Image_Detection/processed_dataset_final/selected_records.csv

TOTAL SELECTED IMAGES : 32,000

Generating Train / Val / Test split...
 real / ffhq             -> train  8000 | val  1000 | test  1000
 real / utkface          -> train  1600 | val   200 | test   200
 real / lfw              -> train  3203 | val   400 | test   397
   ai / stable_diffusion -> tra

train:   0%|          | 0/25604 [00:00<?, ?it/s]

------------------------------------------------------------
TRAIN SUMMARY
Newly Processed : 23,017
Resumed         : 2,587
No face detected: 10,774
Skipped         : 0
Manifest saved to: /content/drive/MyDrive/AI_Image_Detection/processed_dataset_final/train_manifest.csv

PROCESSING VALIDATION SET

Processing VAL Set


val:   0%|          | 0/3205 [00:00<?, ?it/s]

------------------------------------------------------------
VAL SUMMARY
Newly Processed : 2,861
Resumed         : 344
No face detected: 1,329
Skipped         : 0
Manifest saved to: /content/drive/MyDrive/AI_Image_Detection/processed_dataset_final/val_manifest.csv

PROCESSING TEST SET

Processing TEST Set


test:   0%|          | 0/3191 [00:00<?, ?it/s]

------------------------------------------------------------
TEST SUMMARY
Newly Processed : 2,871
Resumed         : 320
No face detected: 1,324
Skipped         : 0
Manifest saved to: /content/drive/MyDrive/AI_Image_Detection/processed_dataset_final/test_manifest.csv

PREPROCESSING COMPLETED SUCCESSFULLY
Train Manifest : [{'path': '/content/drive/MyDrive/AI_Image_Detection/processed_dataset_final/train/real/00000.png', 'label': 'real', 'source': 'ffhq', 'identity': '20918', 'face_detected': True, 'orig_path': '/content/drive/MyDrive/AI_Image_Detection/raw_datasets/real_images_final/ffhq/20918.png'}, {'path': '/content/drive/MyDrive/AI_Image_Detection/processed_dataset_final/train/real/00001.png', 'label': 'real', 'source': 'ffhq', 'identity': '38592', 'face_detected': False, 'orig_path': '/content/drive/MyDrive/AI_Image_Detection/raw_datasets/real_images_final/ffhq/38592.png'}, {'path': '/content/drive/MyDrive/AI_Image_Detection/processed_dataset_final/train/real/00002.png', 'label': 'r